# Comparison of Age Prediction Models

This notebook compares the k-fold cross-validation results across different model configurations:
- MLP encoder vs Transformer encoder
- No masking vs random masking (lambda=1, lambda=2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

In [ ]:
# Define the 6 model configurations
configs = [
    ('MLP/no_masking', 'MLP (no masking)'),
    ('MLP/masking/lambda_1', 'MLP (masking, λ=1)'),
    ('MLP/masking/lambda_2', 'MLP (masking, λ=2)'),
    ('transformer/no_masking', 'Transformer (no masking)'),
    ('transformer/masking/lambda_1', 'Transformer (masking, λ=1)'),
    ('transformer/masking/lambda_2', 'Transformer (masking, λ=2)'),
]

base_dir = Path('.')

In [ ]:
# Load predictions and metrics for each configuration
data = {}
metrics = {}

for folder, name in configs:
    pred_path = base_dir / folder / 'predictions.npy'
    results_path = base_dir / folder / 'results.json'
    
    if pred_path.exists():
        data[name] = np.load(pred_path, allow_pickle=True).item()
        print(f"Loaded {name}: {len(data[name]['pred_log_age'])} samples")
        
        if results_path.exists():
            with open(results_path) as f:
                metrics[name] = json.load(f)['overall_metrics']
    else:
        print(f"Missing: {pred_path}")

In [ ]:
# Create 2x3 subplot grid for Predicted vs True Age
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (folder, name) in enumerate(configs):
    ax = axes[idx]
    
    if name in data:
        pred_log_age = data[name]['pred_log_age']
        true_log_age = data[name]['true_log_age']
        
        # Convert to linear age (Myr)
        pred_age = 10 ** pred_log_age
        true_age = 10 ** true_log_age
        
        # Scatter plot
        ax.scatter(true_age, pred_age, alpha=0.3, s=10)
        
        # 1:1 line
        lims = [min(true_age.min(), pred_age.min()), max(true_age.max(), pred_age.max())]
        ax.plot(lims, lims, 'r--', linewidth=2, label='y=x')
        
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('True Age (Myr)')
        ax.set_ylabel('Predicted Age (Myr)')
        
        # Title with metrics
        if name in metrics:
            r = metrics[name]['correlation']
            mae = metrics[name]['mae_log']
            ax.set_title(f"{name}\nr={r:.3f}, MAE(log)={mae:.3f}")
        else:
            ax.set_title(name)
        
        ax.legend(loc='upper left')
    else:
        ax.text(0.5, 0.5, f"{name}\n(no data)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title(name)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Create 2x3 subplot grid for Residuals
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (folder, name) in enumerate(configs):
    ax = axes[idx]
    
    if name in data:
        pred_log_age = data[name]['pred_log_age']
        true_log_age = data[name]['true_log_age']
        
        # Compute residuals in log space
        residuals = pred_log_age - true_log_age
        
        # Histogram
        ax.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
        ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
        
        ax.set_xlabel('Residual (log age)')
        ax.set_ylabel('Count')
        
        # Title with metrics
        if name in metrics:
            mae = metrics[name]['mae_log']
            rmse = metrics[name]['rmse_log']
            ax.set_title(f"{name}\nMAE={mae:.3f}, RMSE={rmse:.3f}")
        else:
            ax.set_title(name)
        
        ax.legend(loc='upper right')
    else:
        ax.text(0.5, 0.5, f"{name}\n(no data)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title(name)

plt.tight_layout()
plt.savefig('residuals_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table of metrics
print("\nModel Comparison Summary")
print("=" * 80)
print(f"{'Model':<35} {'Corr':>8} {'MAE(log)':>10} {'MAPE':>8} {'1σ calib':>10}")
print("-" * 80)

for folder, name in configs:
    if name in metrics:
        m = metrics[name]
        within_1s = m.get('within_1sigma', float('nan'))
        print(f"{name:<35} {m['correlation']:>8.3f} {m['mae_log']:>10.4f} {m['mape']:>7.1f}% {within_1s:>9.1f}%")
    else:
        print(f"{name:<35} {'N/A':>8}")